# Notebook 2: Candidate Circuit Discovery and Stability

Colab-ready workflow for discovering candidate circuits in a trained flow-circuits encoder and inspecting their spatial structure and stability.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/jacobposchl/model-interpretability.git'
REPO_DIR = Path('/content/model_interpretability' if IN_COLAB else Path.cwd()).resolve()

if IN_COLAB:
    if REPO_DIR.exists():
        subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
    else:
        subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', '.', '-q'], check=True)

    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive/flow_circuits')
    DATA_ROOT  = Path('/content/drive/MyDrive/ctls/data')
else:
    if not (REPO_DIR / 'flow_circuits').exists():
        raise RuntimeError('Run this notebook from the repository root or in Google Colab.')
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR / 'artifacts' / 'local_notebook_runtime'
    DATA_ROOT  = DRIVE_ROOT / 'data'

BACKBONE_ROOT   = DRIVE_ROOT / 'backbones'
EXPERIMENT_ROOT = DRIVE_ROOT / 'experiments'
NOTEBOOK_ROOT   = DRIVE_ROOT / 'notebook_runs'
for path in (DRIVE_ROOT, DATA_ROOT, BACKBONE_ROOT, EXPERIMENT_ROOT, NOTEBOOK_ROOT):
    path.mkdir(parents=True, exist_ok=True)

CONFIG_ROOT = REPO_DIR / 'configs' / 'flow'
print(f"Repo      : {REPO_DIR}")
print(f"Drive root: {DRIVE_ROOT}")
print(f"Data      : {DATA_ROOT}")

In [ ]:
from flow_circuits.data import build_cifar10_splits
from flow_circuits.discovery import CandidateCircuitDiscoverer
from flow_circuits.training import collect_model_outputs, load_components_from_checkpoint

## Step 1 — Configure Your Run

Edit the cell below. Everything else runs automatically.

---

**`TRAINING_MODE`** — controls how long training runs:

| Mode | Epochs | Time on T4 GPU | Use for |
|------|--------|----------------|---------|
| `'smoke'` | 1 per phase | ~2 min | Verifying the pipeline runs end-to-end |
| `'debug'` | 5 per phase | ~30 min | Checking loss curves move in the right direction |
| `'full'` | Config defaults (20+20) | 3–6 hr | Scientifically valid results |

> **Note:** Results from `'smoke'` or `'debug'` are **not** scientifically meaningful — the model has not converged.

---

**`CONFIG_NAME`** — selects the model configuration:

- `'resnet18_base'` — Phases A + B only. No trajectory alignment (Phase C skipped). Faster.
- `'resnet18_aligned'` — Full pipeline with Phase C trajectory alignment sweep.

---

**Path overrides** — set these to reuse files you've already computed. Leave as `None` to auto-generate.

In [ ]:
# ── Training mode ─────────────────────────────────────────────────────────────
#   'smoke' = 1 epoch, ~2 min    (pipeline check only — results not meaningful)
#   'debug' = 5 epochs/phase, ~30 min on GPU  (trends are directionally valid)
#   'full'  = config defaults, 3-6 hr on T4   (scientifically valid results)
TRAINING_MODE = 'smoke'  # <- change me

# ── Model configuration ───────────────────────────────────────────────────────
#   'resnet18_base'    = Phases A+B only (no trajectory alignment)
#   'resnet18_aligned' = Full pipeline with Phase C trajectory alignment sweep
CONFIG_NAME = 'resnet18_aligned'  # <- change me

# ── Reuse existing artifacts (set to None to auto-generate) ───────────────────
BACKBONE_WEIGHTS_PATH = None  # path to a trained ResNet18-CIFAR10 .pt file
CHECKPOINT_PATH       = None  # path to a trained flow-circuits final.pt file
CIRCUITS_PATH         = None  # path to a candidate_circuits.json file

# ── Output directory ──────────────────────────────────────────────────────────
OUTPUT_DIR = str(NOTEBOOK_ROOT / 'nb02')

In [ ]:
import copy
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import torch
import yaml

# ── Training mode settings ────────────────────────────────────────────────────
_MODE_SETTINGS = {
    'smoke': {
        'label': 'Smoke test (~2 min) — pipeline check only, results NOT meaningful',
        'batch_size': 32, 'num_workers': 0,
        'phase_epochs': {'phase_a': 1, 'phase_b': 1, 'phase_c': 1},
        'baseline_fit': 128, 'baseline_eval': 128, 'val_images': 64,
        'align_pairs': 256, 'disc_images': 256, 'disc_bootstrap': 4, 'interv_images': 128,
    },
    'debug': {
        'label': 'Debug run (~30 min on GPU) — trends meaningful, not publication quality',
        'batch_size': 64, 'num_workers': 2,
        'phase_epochs': {'phase_a': 5, 'phase_b': 5, 'phase_c': 5},
        'baseline_fit': 512, 'baseline_eval': 512, 'val_images': 512,
        'align_pairs': 1024, 'disc_images': 2000, 'disc_bootstrap': 10, 'interv_images': 1000,
    },
    'full': {
        'label': 'Full training (3-6 hr on T4) — scientifically valid results',
        'batch_size': None, 'num_workers': None, 'phase_epochs': None,
        'baseline_fit': None, 'baseline_eval': None, 'val_images': None,
        'align_pairs': None, 'disc_images': None, 'disc_bootstrap': None, 'interv_images': None,
    },
}
_ms = _MODE_SETTINGS[TRAINING_MODE]
print(f"Training mode : {TRAINING_MODE}")
print(f"  {_ms['label']}")

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

CLI_MODULES = {
    'flow-train': 'flow_circuits.cli.train',
    'flow-discover': 'flow_circuits.cli.discover',
}

def run_flow_cli(command: str, *args: str) -> None:
    def _stream(cmd):
        import os
        env = os.environ.copy()
        env["PYTHONUNBUFFERED"] = "1"
        process = subprocess.Popen(
            cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, cwd=REPO_DIR, env=env,
        )
        while True:
            line = process.stdout.readline()
            if not line:
                break
            print(line, end="", flush=True)
        process.wait()
        if process.returncode != 0:
            raise subprocess.CalledProcessError(process.returncode, cmd)
    try:
        _stream([command, *args])
    except FileNotFoundError:
        _stream([sys.executable, '-m', CLI_MODULES[command], *args])

with open(str(CONFIG_ROOT / f'{CONFIG_NAME}.yaml'), 'r', encoding='utf-8') as handle:
    config = yaml.safe_load(handle)

if not BACKBONE_WEIGHTS_PATH:
    BACKBONE_WEIGHTS_PATH = str(BACKBONE_ROOT / f"{config['backbone']['arch']}_cifar10_supervised.pt")
config['data']['data_dir'] = str(DATA_ROOT)
config['data']['download'] = False
config['backbone']['weights_path'] = str(BACKBONE_WEIGHTS_PATH)
config['logging']['checkpoint_dir'] = str(EXPERIMENT_ROOT / config['experiment']['name'])

effective_config = copy.deepcopy(config)
effective_phase_epochs = None if _ms['phase_epochs'] is None else copy.deepcopy(_ms['phase_epochs'])
if effective_phase_epochs is not None and effective_config['experiment'].get('mode', 'base') != 'aligned':
    effective_phase_epochs['phase_c'] = 0
if TRAINING_MODE != 'full':
    effective_config['data']['batch_size'] = _ms['batch_size']
    effective_config['data']['num_workers'] = _ms['num_workers']
    effective_config['training']['phase_epochs'] = effective_phase_epochs
    effective_config['training']['baseline_fit_images'] = _ms['baseline_fit']
    effective_config['training']['baseline_eval_images'] = _ms['baseline_eval']
    effective_config['training']['validation_images'] = _ms['val_images']
    effective_config['training']['alignment_max_pairs'] = _ms['align_pairs']
    effective_config['discovery']['max_images'] = _ms['disc_images']
    effective_config['discovery']['bootstrap_iterations'] = _ms['disc_bootstrap']
    effective_config['interventions']['max_images'] = _ms['interv_images']
    effective_config['logging']['checkpoint_dir'] = str(OUTPUT_DIR / f'{TRAINING_MODE}_checkpoints')

EFFECTIVE_CONFIG = OUTPUT_DIR / f'{TRAINING_MODE}_config.yaml'
EFFECTIVE_CONFIG.write_text(yaml.safe_dump(effective_config, sort_keys=False), encoding='utf-8')

PHASE_B_CHECKPOINT = Path(effective_config['logging']['checkpoint_dir']) / 'phase_b.pt'
PHASE_C_CHECKPOINT = Path(effective_config['logging']['checkpoint_dir']) / 'phase_c.pt'
EFFECTIVE_CHECKPOINT = Path(CHECKPOINT_PATH) if CHECKPOINT_PATH else Path(effective_config['logging']['checkpoint_dir']) / 'final.pt'
EFFECTIVE_CIRCUITS = Path(CIRCUITS_PATH) if CIRCUITS_PATH else Path(effective_config['logging']['checkpoint_dir']) / 'candidate_circuits.json'
print(f"Circuits    : {EFFECTIVE_CIRCUITS}")
print(f"\nConfig      : {EFFECTIVE_CONFIG}")
print(f"Phases      : {effective_config['training']['phase_epochs']}")
print(f"Phase B ckpt: {PHASE_B_CHECKPOINT}")
print(f"Phase C ckpt: {PHASE_C_CHECKPOINT}")
print(f"Checkpoint  : {EFFECTIVE_CHECKPOINT}")

## Step 2 — Train and Discover Circuits

The cell below:
1. Trains the flow-circuit encoder (skipped if checkpoint exists)
2. Runs candidate circuit discovery on the discovery split

Discovery clusters the encoder's future descriptors at each (layer, spatial cell) node,
then finds groups of nodes that consistently co-activate across images.

In [ ]:
# ── Step 1: Train flow model (skipped if checkpoint exists) ───────────────────
RESUME_CHECKPOINT = PHASE_B_CHECKPOINT if (not EFFECTIVE_CHECKPOINT.exists() and PHASE_B_CHECKPOINT.exists()) else None
if not EFFECTIVE_CHECKPOINT.exists():
    if RESUME_CHECKPOINT is not None:
        print("Final checkpoint not found — resuming from Phase B checkpoint...")
        print(f"  {RESUME_CHECKPOINT}")
        run_flow_cli('flow-train', '--config', str(EFFECTIVE_CONFIG), '--resume', str(RESUME_CHECKPOINT))
    else:
        print("Flow model checkpoint not found — starting training...")
        run_flow_cli('flow-train', '--config', str(EFFECTIVE_CONFIG))
else:
    print(f"Flow model checkpoint found — skipping training.")
    print(f"  {EFFECTIVE_CHECKPOINT}")

# ── Step 2: Discover circuits (skipped if artifact exists) ────────────────────
if not EFFECTIVE_CIRCUITS.exists():
    print("\nCircuit artifact not found — running discovery...")
    run_flow_cli('flow-discover', '--checkpoint', str(EFFECTIVE_CHECKPOINT), '--output', str(EFFECTIVE_CIRCUITS))
else:
    print(f"\nCircuit artifact found — skipping discovery.")
    print(f"  {EFFECTIVE_CIRCUITS}")

circuits_artifact = json.loads(EFFECTIVE_CIRCUITS.read_text(encoding='utf-8'))

n_circuits = len(circuits_artifact['circuits'])
print(f"\nDiscovery complete: {n_circuits} candidate circuit(s) found.")
if n_circuits == 0:
    print("  No circuits passed the stability and engagement filters.")
    print("  Try running with TRAINING_MODE='full' for a properly trained model.")

## Step 3 — Inspect Discovered Circuits

Each circuit is a group of (layer, spatial cell) nodes that tend to co-activate for
the same set of images. The preview below shows the key properties of each circuit.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
components = load_components_from_checkpoint(EFFECTIVE_CHECKPOINT, device=device)
loaders = build_cifar10_splits(
    data_dir=components.config['data']['data_dir'],
    batch_size=components.config['data']['batch_size'],
    num_workers=components.config['data'].get('num_workers', 0),
    seed=components.config['data'].get('seed', 0),
    augment_fit=components.config['data'].get('augment_fit', True),
    download=components.config['data'].get('download', True),
)
discovery_outputs = collect_model_outputs(
    components,
    loaders['discovery'],
    device=device,
    max_images=components.config['discovery'].get('max_images'),
)

n_total = circuits_artifact['metadata']['n_images']
n_circuits = len(circuits_artifact['circuits'])
print(f"Discovery set: {n_total} images")
print(f"Circuits found: {n_circuits}")
print()
if n_circuits == 0:
    print("No circuits to display.")
else:
    print(f"  {'ID':>3}  {'Images':>7}  {'Nodes':>6}  {'Repr. node':>12}  {'Stability':>10}")
    print(f"  {'-'*3}  {'-'*7}  {'-'*6}  {'-'*12}  {'-'*10}")
    for c in circuits_artifact['circuits'][:10]:
        rep = c['representative_node']
        stab = c.get('stability', {}).get('mean_cluster_stability', float('nan'))
        print(
            f"  {c['id']:>3}  {len(c['image_set']):>7}  {len(c['active_nodes']):>6}"
            f"  layer={rep[0]} cell={rep[1]:>2}  {stab:>10.3f}"
        )
    if n_circuits > 10:
        print(f"  ... and {n_circuits - 10} more circuits.")

In [ ]:
grid_size = circuits_artifact['metadata']['grid_size']
max_plots = min(4, len(circuits_artifact['circuits']))
if max_plots == 0:
    print('No candidate circuits were discovered for this artifact.')
else:
    fig, axes = plt.subplots(1, max_plots, figsize=(4 * max_plots, 3.5))
    if max_plots == 1:
        axes = [axes]
    for axis, circuit in zip(axes, circuits_artifact['circuits'][:max_plots]):
        heatmap = __import__('numpy').zeros((grid_size, grid_size), dtype=float)
        for _, cell_idx in circuit['active_nodes']:
            row = cell_idx // grid_size
            col = cell_idx % grid_size
            heatmap[row, col] += 1.0
        image = axis.imshow(heatmap, cmap='magma')
        axis.set_title(f"Circuit {circuit['id']}\n{len(circuit['active_nodes'])} active nodes")
        plt.colorbar(image, ax=axis, fraction=0.046, pad=0.04)
    plt.suptitle('Spatial footprint of each circuit (each cell = one grid position)', y=1.02)
    plt.tight_layout()

In [ ]:
import numpy as np
all_nodes = set()
purities = []
engagement_sizes = []
for circuit in circuits_artifact['circuits']:
    for node in circuit['active_nodes']:
        all_nodes.add(tuple(node))
    if circuit.get('purity') is not None:
        purities.append(float(circuit['purity']))
    engagement_sizes.append(len(circuit['engagement_profile']))

n_total_nodes = circuits_artifact['metadata']['n_layers'] * circuits_artifact['metadata']['n_cells']
coverage = len(all_nodes) / max(1, n_total_nodes)
print(f"Active node coverage : {len(all_nodes)}/{n_total_nodes} nodes ({coverage:.1%})")
print(f"  (How many distinct (layer, cell) positions appear in at least one circuit)")
if purities:
    print(f"\nPurity range : {min(purities):.3f} - {max(purities):.3f}")
    print(f"  (Fraction of a circuit's images that share the same class label)")
if engagement_sizes:
    print(f"\nMean engaged nodes per circuit : {np.mean(engagement_sizes):.1f}")
    print(f"  (Nodes with high cosine between predicted and actual next-layer flow)")

In [ ]:
stability_dir = OUTPUT_DIR / 'stability'
stability_files = sorted(stability_dir.glob('*.json'))
if not stability_files:
    print('No extra stability artifacts found under', stability_dir)
else:
    stability_counts = []
    for path in stability_files:
        artifact = json.loads(path.read_text(encoding='utf-8'))
        stability_counts.append({'file': path.name, 'n_circuits': len(artifact['circuits'])})
    stability_counts